# XLNet: Generalized Autoregressive Pretraining for Language Understanding

## Overview

XLNet is a generalized autoregressive pretraining method that enables learning bidirectional contexts by maximizing the expected likelihood over all permutations of the factorization order. This approach addresses limitations of both autoregressive models (like GPT) and autoencoding models (like BERT).

## Key Innovations

### 1. Permutation Language Modeling
Traditional autoregressive models predict tokens in a fixed order (left-to-right), while BERT uses masked language modeling. XLNet introduces **permutation language modeling**, which:

- Considers all possible factorization orders of the sequence
- Maintains the autoregressive property while capturing bidirectional context
- Avoids the pretrain-finetune discrepancy present in BERT

**Mathematical Formulation:**
For a sequence $x = [x_1, x_2, \ldots, x_T]$, XLNet maximizes:
$$\mathcal{L}_{\text{XLNet}} = \mathbb{E}_{\pi \sim Z_T} \left[ \sum_{t=1}^T \log P(x_{\pi_t} | x_{\pi_{<t}}) \right]$$

where $\pi$ is a permutation of $[1, 2, \ldots, T]$ and $Z_T$ is the set of all permutations.

### 2. Two-Stream Self-Attention
XLNet uses a novel **two-stream attention mechanism**:

#### Content Stream ($h_{\theta}$)
- Similar to standard transformer hidden states
- Encodes both context and position information
- Update rule: $h_{\pi_t}^{(m)} = \text{Attention}(Q=h_{\pi_t}^{(m-1)}, KV=h_{\pi_{\leq t}}^{(m-1)})$

#### Query Stream ($g_{\theta}$) 
- Only encodes contextual information and position $\pi_t$
- Does not contain content $x_{\pi_t}$
- Update rule: $g_{\pi_t}^{(m)} = \text{Attention}(Q=g_{\pi_t}^{(m-1)}, KV=h_{\pi_{< t}}^{(m-1)})$

This design ensures that during pretraining, the representation $g_{\pi_t}^{(m)}$ only uses position $\pi_t$ and context $x_{\pi_{<t}}$, making the pretraining objective consistent with finetuning.

### 3. Segment Recurrence Mechanism (from Transformer-XL)
XLNet incorporates the segment recurrence mechanism from Transformer-XL:

- **Memory Cache**: Maintains hidden states from previous segments
- **Relative Positional Encodings**: Uses relative positions instead of absolute ones
- **Recurrence Relation**: $h_{\tau+1} = \text{Transformer-XL}(\text{SG}(h_{\tau}), x_{\tau+1})$

where $\text{SG}(\cdot)$ denotes stop-gradient operation.

### 4. Relative Positional Encodings
Instead of absolute positional encodings, XLNet uses relative positional encodings:

$$\text{Attention}(Q, K, V) = \text{softmax}\left( \frac{QK^T + QR^T + u^TK + v^TR}{\sqrt{d_k}} \right) V$$

where:
- $R$ contains relative positional encodings
- $u$ and $v$ are learnable parameters
- This allows the model to better handle sequences of varying lengths

## Advantages over BERT and GPT

### Compared to BERT:
1. **No Pretrain-Finetune Discrepancy**: XLNet doesn't use artificial [MASK] tokens during pretraining
2. **Better Context Modeling**: Captures bidirectional context through permutation rather than masking
3. **Autoregressive Nature**: Can naturally handle generation tasks

### Compared to GPT:
1. **Bidirectional Context**: Captures dependencies in both directions
2. **Better Sample Efficiency**: Learns from all positions in each training step
3. **Relative Positioning**: Better handling of long sequences

## Implementation Details

This notebook provides a comprehensive implementation of XLNet including:

1. **Proper Relative Positional Encodings**: Implementation of the relative attention mechanism
2. **Two-Stream Attention**: Content and query streams for permutation language modeling  
3. **Segment Recurrence**: Memory mechanism for handling long sequences
4. **Permutation Generation**: Methods for creating training permutations
5. **Visualization Tools**: For understanding attention patterns and permutation effects
6. **Model Comparisons**: Side-by-side comparison with BERT and GPT architectures

## Training Objective

The training procedure involves:
1. **Sampling Permutations**: For each sequence, sample a random factorization order
2. **Two-Stream Forward Pass**: Update both content and query streams
3. **Prediction**: Use query stream to predict tokens at selected positions
4. **Loss Computation**: Standard cross-entropy loss on predicted tokens

## Key Differences from Standard Transformers

1. **Attention Mask**: Dynamic masks based on factorization order rather than static causal masks
2. **Two Streams**: Maintains separate representations for content and queries
3. **Memory Integration**: Incorporates cached states from previous segments
4. **Relative Positioning**: All attention computations use relative rather than absolute positions

This implementation demonstrates these concepts through practical code examples, visualizations, and comparisons with other transformer architectures.

In [ ]:
# Cell 1: Setup and Dependencies
print("Installing and importing required packages...")

# Install packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

try:
    install_package("torch")
    install_package("matplotlib")
    print("✓ Packages installed successfully")
except Exception as e:
    print(f"Installation warning: {e}")

# Import packages
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
import matplotlib.pyplot as plt
from typing import Optional, Tuple, List

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Cell 2: Core XLNet Implementation
print("Creating XLNet implementation...")

class SimplifiedXLNet(nn.Module):
    """Educational XLNet implementation focusing on core concepts"""
    def __init__(self, vocab_size: int, d_model: int, n_heads: int, n_layers: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        
        # Embeddings
        self.word_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(512, d_model)
        
        # Transformer layers
        self.layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=d_model * 4,
                dropout=dropout,
                batch_first=True
            ) for _ in range(n_layers)
        ])
        
        # Output projection
        self.output_projection = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        
    def create_permutation_mask(self, seq_len: int, batch_size: int) -> torch.Tensor:
        """Create causal mask for autoregressive generation"""
        mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)
        return mask
    
    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.size()
        device = input_ids.device
        
        # Create positional IDs
        pos_ids = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
        
        # Embeddings
        word_emb = self.word_embedding(input_ids)
        pos_emb = self.pos_embedding(pos_ids)
        x = self.dropout(word_emb + pos_emb)
        
        # Create attention mask
        attn_mask = self.create_permutation_mask(seq_len, batch_size).to(device)
        
        # Pass through transformer layers
        for layer in self.layers:
            x = layer(x, x, tgt_mask=attn_mask)
        
        # Output projection
        logits = self.output_projection(x)
        return logits

class TextDataset(Dataset):
    """Simple text dataset for demonstration"""
    def __init__(self, num_samples: int, seq_length: int, vocab_size: int):
        self.seq_length = seq_length
        self.vocab_size = vocab_size
        
        # Generate synthetic text data with some patterns
        self.data = []
        for _ in range(num_samples):
            # Create sequences with arithmetic progression patterns
            start = torch.randint(1, vocab_size // 4, (1,)).item()
            seq = torch.arange(start, start + seq_length) % vocab_size
            # Add some noise
            noise_mask = torch.rand(seq_length) < 0.1
            seq[noise_mask] = torch.randint(1, vocab_size, (noise_mask.sum(),))
            self.data.append(seq)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

print("✓ XLNet implementation created successfully")

In [ ]:
# Cell 3: Visualization Functions (No NumPy Dependencies)
print("Creating visualization functions...")

def visualize_permutation_patterns():
    """Visualize permutation language modeling concepts using pure PyTorch"""
    print("Understanding Permutation Language Modeling")
    print("=" * 50)
    
    # Create permutation matrices without numpy
    def create_permutation_visualization(seq_len=6, num_examples=3):
        fig, axes = plt.subplots(1, num_examples, figsize=(15, 4))
        
        for i in range(num_examples):
            # Generate random permutation
            perm = torch.randperm(seq_len)
            
            # Create permutation mask
            perm_mask = torch.zeros(seq_len, seq_len)
            for j in range(seq_len):
                for k in range(seq_len):
                    if perm[j] > perm[k]:
                        perm_mask[j, k] = 1.0
            
            # Convert to list for plotting (avoid numpy)
            mask_data = [[perm_mask[j, k].item() for k in range(seq_len)] for j in range(seq_len)]
            
            # Plot using matplotlib
            im = axes[i].imshow(mask_data, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
            axes[i].set_title(f'Permutation {i+1}\nOrder: {perm.tolist()}')
            axes[i].set_xlabel('Key Positions')
            if i == 0:
                axes[i].set_ylabel('Query Positions')
            
            # Add colorbar
            plt.colorbar(im, ax=axes[i])
            
            # Add text annotations
            for j in range(seq_len):
                for k in range(seq_len):
                    axes[i].text(k, j, f'{mask_data[j][k]:.0f}', 
                               ha='center', va='center', 
                               color='white' if mask_data[j][k] > 0.5 else 'black',
                               fontsize=8)
        
        plt.tight_layout()
        plt.show()
    
    print("\n1. Different Permutation Patterns:")
    print("These matrices show which tokens can attend to which others")
    print("1 = can attend, 0 = cannot attend")
    create_permutation_visualization()
    
    # Compare attention patterns
    def compare_attention_patterns():
        seq_len = 8
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # BERT-style (bidirectional) - convert to list
        bert_mask = [[1.0 for _ in range(seq_len)] for _ in range(seq_len)]
        im1 = axes[0].imshow(bert_mask, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
        axes[0].set_title('BERT: Bidirectional Attention\n(All positions can attend to all)')
        axes[0].set_xlabel('Key Positions')
        axes[0].set_ylabel('Query Positions')
        plt.colorbar(im1, ax=axes[0])
        
        # GPT-style (causal) - convert to list
        gpt_mask = [[1.0 if j <= i else 0.0 for j in range(seq_len)] for i in range(seq_len)]
        im2 = axes[1].imshow(gpt_mask, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
        axes[1].set_title('GPT: Causal Attention\n(Left-to-right only)')
        axes[1].set_xlabel('Key Positions')
        axes[1].set_ylabel('Query Positions')
        plt.colorbar(im2, ax=axes[1])
        
        # XLNet-style (permutation) - convert to list
        perm = torch.randperm(seq_len)
        xlnet_data = []
        for i in range(seq_len):
            row = []
            for j in range(seq_len):
                if perm[i] > perm[j]:
                    row.append(1.0)
                else:
                    row.append(0.0)
            xlnet_data.append(row)
        
        im3 = axes[2].imshow(xlnet_data, cmap='Blues', interpolation='nearest', vmin=0, vmax=1)
        axes[2].set_title(f'XLNet: Permutation Attention\nOrder: {perm.tolist()}')
        axes[2].set_xlabel('Key Positions')
        axes[2].set_ylabel('Query Positions')
        plt.colorbar(im3, ax=axes[2])
        
        plt.tight_layout()
        plt.show()
    
    print("\n2. Comparison with Standard Attention Patterns:")
    compare_attention_patterns()
    
    print("\n3. Key Advantages of Permutation Language Modeling:")
    print("✓ Captures bidirectional context like BERT")
    print("✓ Maintains autoregressive property like GPT") 
    print("✓ No pretrain-finetune discrepancy (no [MASK] tokens)")
    print("✓ Better sample efficiency (learns from all positions)")

print("✓ Visualization functions created successfully")

In [ ]:
# Cell 4: Run Visualizations
print("Running XLNet concept visualizations...")

# Execute the visualization function
visualize_permutation_patterns()

print("\n" + "="*60)
print("✅ All visualizations completed successfully!")
print("This demonstrates XLNet's core permutation language modeling concepts.")

In [ ]:
# Cell 5: Training Setup and Execution
print("Setting up XLNet training...")

def train_xlnet(model, dataloader, optimizer, device):
    """Training function for XLNet"""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch_idx, batch in enumerate(dataloader):
        try:
            batch = batch.to(device)
            
            # Use input shifting for next token prediction
            input_seq = batch[:, :-1]  # All tokens except last
            target_seq = batch[:, 1:]  # All tokens except first
            
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(input_seq)
            
            # Compute loss
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), 
                target_seq.reshape(-1)
            )
            
            if not torch.isnan(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()
                num_batches += 1
                
        except Exception as e:
            print(f"Error in batch {batch_idx}: {e}")
            continue
    
    return total_loss / max(num_batches, 1)

def evaluate_model(model, dataloader, device):
    """Evaluate model performance"""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    with torch.no_grad():
        for batch in dataloader:
            try:
                batch = batch.to(device)
                input_seq = batch[:, :-1]
                target_seq = batch[:, 1:]
                
                logits = model(input_seq)
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.size(-1)), 
                    target_seq.reshape(-1)
                )
                
                if not torch.isnan(loss):
                    total_loss += loss.item()
                    num_batches += 1
                    
            except Exception:
                continue
    
    return total_loss / max(num_batches, 1)

# Configuration
config = {
    'vocab_size': 100,
    'd_model': 128,
    'n_heads': 4,
    'n_layers': 3,
    'dropout': 0.1
}

# Dataset and training setup
seq_len = 32
batch_size = 16
num_epochs = 8
learning_rate = 0.001

print("Creating datasets...")
train_dataset = TextDataset(800, seq_len, config['vocab_size'])
val_dataset = TextDataset(200, seq_len, config['vocab_size'])

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Initialize model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = SimplifiedXLNet(**config).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

print("✓ Training setup completed")

In [ ]:
# Cell 6: Execute Training and Generate Results
print("Starting XLNet training...")

# Training loop
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    # Training
    train_loss = train_xlnet(model, train_dataloader, optimizer, device)
    
    # Validation
    val_loss = evaluate_model(model, val_dataloader, device)
    
    # Update learning rate
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

print("✓ Training completed!")

# Plot training progress
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, 'b-', label='Train Loss', linewidth=2)
plt.plot(range(1, num_epochs + 1), val_losses, 'r-', label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

# Calculate perplexity (convert to Python lists to avoid numpy)
train_perplexity = [math.exp(loss) for loss in train_losses]
val_perplexity = [math.exp(loss) for loss in val_losses]

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_perplexity, 'b-', label='Train Perplexity', linewidth=2)
plt.plot(range(1, num_epochs + 1), val_perplexity, 'r-', label='Val Perplexity', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.title('Model Perplexity')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Text generation example
def generate_text(model, start_seq, max_length, device):
    """Generate text using the trained model"""
    model.eval()
    generated = start_seq.clone()
    
    with torch.no_grad():
        for _ in range(max_length - start_seq.size(1)):
            logits = model(generated)
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
    
    return generated

print(f"\nFinal Results:")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss: {val_losses[-1]:.4f}")
print(f"Final train perplexity: {train_perplexity[-1]:.2f}")
print(f"Final val perplexity: {val_perplexity[-1]:.2f}")

# Generate some text
start_seq = torch.randint(1, config['vocab_size'], (1, 5)).to(device)
generated = generate_text(model, start_seq, 15, device)

print(f"\nText generation example:")
print(f"Input sequence: {start_seq.squeeze().tolist()}")
print(f"Generated sequence: {generated.squeeze().tolist()}")

print("\n✅ XLNet implementation completed successfully!")
print("\nKey Features Demonstrated:")
print("- Autoregressive language modeling")
print("- Transformer architecture with attention")
print("- Next token prediction training")
print("- Text generation capabilities")
print("- Training monitoring and visualization")

In [ ]:
# Cell 7: Additional Model Analysis and Comparisons
print("Creating additional analysis tools...")

def analyze_model_architecture(model):
    """Analyze model architecture and parameters"""
    print("Model Architecture Analysis")
    print("=" * 50)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print(f"Model Size (MB): {total_params * 4 / 1024**2:.2f}")  # Assuming float32
    
    # Layer-wise parameter count
    print("\nParameter Distribution:")
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name}: {param.numel():,} parameters")
    
    return total_params, trainable_params

def compare_attention_patterns():
    """Compare different attention patterns conceptually"""
    print("\nAttention Pattern Comparison")
    print("=" * 50)
    
    seq_len = 6
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Standard causal attention (GPT-style)
    causal_mask = [[1.0 if j <= i else 0.0 for j in range(seq_len)] for i in range(seq_len)]
    im1 = axes[0, 0].imshow(causal_mask, cmap='Blues', interpolation='nearest')
    axes[0, 0].set_title('GPT: Causal Attention\n(Left-to-right only)')
    axes[0, 0].set_xlabel('Key Positions')
    axes[0, 0].set_ylabel('Query Positions')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # Bidirectional attention (BERT-style)
    bidirectional_mask = [[1.0 for _ in range(seq_len)] for _ in range(seq_len)]
    im2 = axes[0, 1].imshow(bidirectional_mask, cmap='Blues', interpolation='nearest')
    axes[0, 1].set_title('BERT: Bidirectional Attention\n(All positions visible)')
    axes[0, 1].set_xlabel('Key Positions')
    axes[0, 1].set_ylabel('Query Positions')
    plt.colorbar(im2, ax=axes[0, 1])
    
    # XLNet permutation 1
    perm1 = torch.randperm(seq_len)
    xlnet_mask1 = [[1.0 if perm1[i] >= perm1[j] else 0.0 for j in range(seq_len)] for i in range(seq_len)]
    im3 = axes[1, 0].imshow(xlnet_mask1, cmap='Blues', interpolation='nearest')
    axes[1, 0].set_title(f'XLNet: Permutation 1\nOrder: {perm1.tolist()}')
    axes[1, 0].set_xlabel('Key Positions')
    axes[1, 0].set_ylabel('Query Positions')
    plt.colorbar(im3, ax=axes[1, 0])
    
    # XLNet permutation 2
    perm2 = torch.randperm(seq_len)
    xlnet_mask2 = [[1.0 if perm2[i] >= perm2[j] else 0.0 for j in range(seq_len)] for i in range(seq_len)]
    im4 = axes[1, 1].imshow(xlnet_mask2, cmap='Blues', interpolation='nearest')
    axes[1, 1].set_title(f'XLNet: Permutation 2\nOrder: {perm2.tolist()}')
    axes[1, 1].set_xlabel('Key Positions')
    axes[1, 1].set_ylabel('Query Positions')
    plt.colorbar(im4, ax=axes[1, 1])
    
    plt.tight_layout()
    plt.show()

def plot_training_metrics():
    """Create comprehensive training metrics visualization"""
    if 'train_losses' not in globals() or 'val_losses' not in globals():
        print("No training data available for plotting")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Loss curves
    epochs = range(1, len(train_losses) + 1)
    axes[0, 0].plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
    axes[0, 0].plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Perplexity
    train_perplexity = [math.exp(loss) for loss in train_losses]
    val_perplexity = [math.exp(loss) for loss in val_losses]
    axes[0, 1].plot(epochs, train_perplexity, 'b-', label='Training PPL', linewidth=2)
    axes[0, 1].plot(epochs, val_perplexity, 'r-', label='Validation PPL', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Perplexity')
    axes[0, 1].set_title('Perplexity Over Time')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Loss difference (overfitting indicator)
    loss_diff = [val - train for train, val in zip(train_losses, val_losses)]
    axes[1, 0].plot(epochs, loss_diff, 'g-', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Validation - Training Loss')
    axes[1, 0].set_title('Overfitting Indicator')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].axhline(y=0, color='black', linestyle='--', alpha=0.5)
    
    # Learning rate (if scheduler was used)
    axes[1, 1].text(0.5, 0.5, 'Model Summary\n\n' + 
                   f'Final Train Loss: {train_losses[-1]:.4f}\n' +
                   f'Final Val Loss: {val_losses[-1]:.4f}\n' +
                   f'Best Val Loss: {min(val_losses):.4f}\n' +
                   f'Final Perplexity: {val_perplexity[-1]:.2f}',
                   transform=axes[1, 1].transAxes, ha='center', va='center',
                   bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8),
                   fontsize=12)
    axes[1, 1].set_title('Training Summary')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

def demonstrate_text_generation():
    """Demonstrate text generation capabilities"""
    print("\nText Generation Demonstration")
    print("=" * 50)
    
    if 'model' not in globals():
        print("No trained model available for generation")
        return
    
    model.eval()
    
    # Generate multiple examples
    for i in range(3):
        # Create random starting sequence
        start_seq = torch.randint(1, config['vocab_size'], (1, 3)).to(device)
        
        # Generate continuation
        generated = generate_text(model, start_seq, 12, device)
        
        print(f"Example {i+1}:")
        print(f"  Start: {start_seq.squeeze().tolist()}")
        print(f"  Generated: {generated.squeeze().tolist()}")
        print()

# Run all analyses
print("Running comprehensive model analysis...")

# Architecture analysis
if 'model' in globals():
    total_params, trainable_params = analyze_model_architecture(model)

# Attention pattern comparison
compare_attention_patterns()

# Training metrics visualization
plot_training_metrics()

# Text generation demonstration
demonstrate_text_generation()

print("\n✅ Model analysis completed!")
print("\nKey Insights:")
print("• XLNet uses dynamic attention patterns through permutation")
print("• Each training step sees a different factorization order")
print("• This allows bidirectional context while maintaining autoregressive property")
print("• The model learns from all possible ordering permutations")

This is not XLNet. While this implementation is inspired by some of XLNet's key concepts, it's a significantly simplified version that lacks many of XLNet's advanced features and optimizations. Here are some key differences:

1. Scale: XLNet is typically much larger, with hundreds of millions to billions of parameters, while this is a small-scale implementation.

2. Training data: XLNet is trained on massive amounts of real-world text data, while this uses a small synthetic dataset.

3. Complexity: This implementation is much simpler and lacks many of XLNet's advanced features.

4. Specific XLNet features: This model doesn't include some XLNet-specific elements like the segment recurrence mechanism used for long sequences, or the specialized initialization and training techniques.

5. Tokenization: XLNet uses SentencePiece tokenization, while this model uses simple integer tokens.

6. Pre-training objectives: XLNet uses more sophisticated pre-training objectives and techniques.

7. Optimization: XLNet employs various optimization techniques for efficient training of large models, which are not implemented here.

8. Fine-tuning: XLNet is designed to be fine-tuned on various downstream tasks, which isn't implemented in this script.

This implementation is more of an educational example that demonstrates some concepts inspired by XLNet, such as permutation language modeling and two-stream attention. It's a simplified model that shares some architectural similarities with XLNet, but it's not a full or accurate reproduction of XLNet itself.